In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/jabodetabek_house_price.csv")

In [3]:
df_clean = df.drop(columns=["hrg_m2_tanah", "hrg_m2_bgn", "rasio"], errors="ignore").copy()
log = []

def catat(langkah, sebelum, sesudah):
    log.append({"langkah": langkah, "terbuang": sebelum - sesudah, "sisa": sesudah})

In [4]:
kolom_cek = [c for c in df_clean.columns if c not in ["url", "ads_id"]]

n = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=kolom_cek, keep="first")
catat("1. duplikat", n, len(df_clean))

In [5]:
kw_komersial = r"\b(?:kost|kos|kostan|kosan)\b|hotel|guest ?house|penginapan|ex kantor|ex salon|ex restoran"
dilindungi = [3127, 1709, 3446]

mask = df_clean["title"].str.contains(kw_komersial, case=False, na=False) & ~df_clean.index.isin(dilindungi)
print(mask.sum())
df_clean[mask][["title", "bedrooms", "price_in_rp"]]

20


,title,bedrooms,price_in_rp
346,Di Jual Cepat Rumah Kost kostan di Rawalumbu B...,36.0,9.000000e+09
1333,"Rumah Kos di Dramaga, Dekat Kampus Ipb, Bogor",37.0,2.700000e+09
1371,Jual Villa 4600 M Lokasi Cipanas Depan Hotel A...,4.0,7.000000e+09
1372,Jual Villa Ex Restoran Luas Tanah 1500 Meter S...,3.0,2.500000e+09
1392,Hotel di Kawasan Pariwisata Dan Masih Aktif di...,13.0,4.500000e+10
2007,Chandra Rumah Kos Baru Uk 8x17m Lokasi Bagus d...,10.0,5.500000e+09
2030,Dhyana Rumah Kos Baru Uk 8x17m Lokasi Bagus di...,10.0,5.500000e+09
2282,"Rumah Kost Exclusive 21 Kt di Cipete, Ala Hote...",21.0,2.600000e+10
2283,Guest House 20 Kamar di MPR Gaharu Cipete,20.0,2.600000e+10
2303,Rumah 3 Lantai Ex Kantor Bagus Siap Huni di Ke...,12.0,1.500000e+10


In [6]:
n = len(df_clean)
df_clean = df_clean[~mask]
catat("2. properti komersial", n, len(df_clean))

In [7]:
kw_oper = r"oper kredit|over kredit|overkredit|take over|oper murah"
m_oper = df_clean["title"].str.contains(kw_oper, case=False, na=False)
print(m_oper.sum())
n = len(df_clean)
df_clean = df_clean[~m_oper]
catat("3a. oper kredit", n, len(df_clean))

m_lelang = df_clean["title"].str.contains("lelang", case=False, na=False)
print(m_lelang.sum())
n = len(df_clean)
df_clean = df_clean[~m_lelang]
catat("3b. lelang", n, len(df_clean))

10
5


In [8]:
# 4a. luas bangunan kosong
n = len(df_clean)
df_clean = df_clean[df_clean["building_size_m2"].notna()]
catat("4a. bangunan kosong", n, len(df_clean))

# 4b. luas bangunan < 10 m2
n = len(df_clean)
df_clean = df_clean[df_clean["building_size_m2"] >= 10]
catat("4b. bangunan < 10 m2", n, len(df_clean))

# 4c. bangunan kecil tapi harga/m2 tidak wajar
m = (df_clean["building_size_m2"] < 30) & (df_clean["price_in_rp"] / df_clean["building_size_m2"] > 100e6)
print(list(df_clean[m].index))          # harus 3060 dan 3333
n = len(df_clean)
df_clean = df_clean[~m]
catat("4c. bangunan kecil harga tak wajar", n, len(df_clean))

# 4d. kamar tidur >= 10 dan kamar mandi >= kamar tidur (pola kos)
m = (df_clean["bedrooms"] >= 10) & (df_clean["bathrooms"] >= df_clean["bedrooms"])
print(list(df_clean[m].index))          # harus 1370, 2305, 2355, 2357, 2642, 2659
n = len(df_clean)
df_clean = df_clean[~m]
catat("4d. pola kos (kamar>=10, KM>=kamar)", n, len(df_clean))

# 4e. keputusan manual per baris
manual = [2906, 21, 269, 1455]
n = len(df_clean)
df_clean = df_clean.drop(index=manual, errors="ignore")
catat("4e. manual (2906, 21, 269, 1455)", n, len(df_clean))

[3060, 3333]
[1370, 2305, 2355, 2357, 2642, 2659]


In [9]:
df_clean = df_clean.drop(columns=["lat", "long", "year_built"])

pd.DataFrame(log)

,langkah,terbuang,sisa
0,1. duplikat,79,3474
1,2. properti komersial,20,3454
2,3a. oper kredit,10,3444
3,3b. lelang,5,3439
4,4a. bangunan kosong,2,3437
5,4b. bangunan < 10 m2,1,3436
6,4c. bangunan kecil harga tak wajar,2,3434
7,"4d. pola kos (kamar>=10, KM>=kamar)",6,3428
8,"4e. manual (2906, 21, 269, 1455)",4,3424


In [10]:
print("Terbuang:", len(df) - len(df_clean), "| sisa:", len(df_clean))
df_clean[["price_in_rp", "land_size_m2", "building_size_m2", "bedrooms", "bathrooms"]].describe().T

Terbuang: 129 | sisa: 3424


,count,mean,std,min,25%,50%,75%,max
price_in_rp,3424.0,3.921589e+09,9.706336e+09,70000000.0,800000000.0,1.500000e+09,3.500000e+09,2.500000e+11
land_size_m2,3424.0,1.949863e+02,3.662448e+02,12.0,74.0,1.060000e+02,1.812500e+02,8.000000e+03
building_size_m2,3424.0,1.799010e+02,2.157945e+02,19.0,65.0,1.100000e+02,2.000000e+02,5.000000e+03
bedrooms,3393.0,3.189803e+00,1.102870e+00,1.0,2.0,3.000000e+00,4.000000e+00,1.600000e+01
bathrooms,3398.0,2.492348e+00,1.214099e+00,1.0,2.0,2.000000e+00,3.000000e+00,9.000000e+00


In [12]:
df_clean.shape

(3424, 24)

In [14]:
tukar = df_clean["furnishing"] == "baru"
print(tukar.sum())   # 27
df_clean[tukar][["title", "property_condition", "furnishing"]]   # lihat dulu

df_clean.loc[tukar, ["property_condition", "furnishing"]] = df_clean.loc[tukar, ["furnishing", "property_condition"]].values

27


In [15]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv("../data/processed/rumah_clean.csv", index=False)